# Tessera Portable Validation Console

The code cell below uses Colab's form-style presentation to keep the implementation hidden by default. Use **Show code** only when needed. The report can be downloaded after a run.

Run the console cell once to display controls. Only execute trusted repositories and commands. Colab Secrets are not isolated from code executing in the same runtime. Linux tests do not validate native Windows hook behavior.


In [ ]:
#@title Tessera Portable Validation Console { display-mode: "form" }
import os, pathlib, subprocess, shutil, tempfile, time, json, re, html, platform, datetime
import ipywidgets as W
from IPython.display import display, FileLink, clear_output

try:
    from google.colab import userdata
except Exception:
    userdata = None

# Responsive native widgets; avoid wide HTML/CSS tables.
def full(widget):
    widget.layout.width = "100%"
    return widget

repo = full(W.Text(value="https://github.com/Jue-dev/Tessera.git", description="Repo", placeholder="HTTPS GitHub URL"))
ref = full(W.Text(value="issue-6/unwind-lifecycle", description="Ref"))
secret_name = full(W.Text(value="GH_TOKEN", description="Colab Secret"))
git_user = full(W.Text(value="x-access-token", description="Git username"))
clone_timeout = W.IntText(value=240, description="Clone sec", layout=W.Layout(width="150px"))
test_timeout = W.IntText(value=900, description="Test sec", layout=W.Layout(width="150px"))
depth = W.IntText(value=1, description="Depth", layout=W.Layout(width="130px"))
keep_workspace = W.Checkbox(value=False, description="Keep workspace")
continue_on_error = W.Checkbox(value=True, description="Continue after test failure")
setup_rows, test_rows = [], []
setup_box, test_box = W.VBox(), W.VBox()

def refresh_setup():
    setup_box.children = tuple(x["row"] for x in setup_rows)
def add_setup(value=""):
    field=full(W.Text(value=value,placeholder="Setup command (trusted code)"))
    remove=W.Button(description="Remove",layout=W.Layout(width="90px"))
    row=W.VBox([field,remove],layout=W.Layout(width="100%"))
    item={"field":field,"row":row}
    remove.on_click(lambda _,item=item: (setup_rows.remove(item),refresh_setup()))
    setup_rows.append(item); refresh_setup()

def refresh_tests():
    test_box.children = tuple(x["row"] for x in test_rows)
def add_test(name="",command="",needs=""):
    namef=full(W.Text(value=name,placeholder="Test name"))
    commandf=full(W.Textarea(value=command,placeholder="Command",layout=W.Layout(width="100%",height="90px")))
    needsf=full(W.Text(value=needs,placeholder="Required tools (comma-separated)"))
    remove=W.Button(description="Remove",layout=W.Layout(width="90px"))
    row=W.VBox([namef,commandf,needsf,remove],layout=W.Layout(width="100%",border="1px solid #ddd",padding="8px"))
    item={"name":namef,"command":commandf,"needs":needsf,"row":row}
    remove.on_click(lambda _,item=item: (test_rows.remove(item),refresh_tests()))
    test_rows.append(item); refresh_tests()

add_setup()
add_test("GCC strict mock","bash verify/windows_mock/run_windows_mock_tests.sh")
add_test("Clang strict mock","EXTRA_CXXFLAGS='-stdlib=libc++' CXX=clang++ bash verify/windows_mock/run_windows_mock_tests.sh","clang++")
add_test("GCC ASan + UBSan","EXTRA_CXXFLAGS='-fsanitize=address,undefined -fno-omit-frame-pointer' ASAN_OPTIONS='detect_leaks=1:halt_on_error=1' UBSAN_OPTIONS='halt_on_error=1:print_stacktrace=1' bash verify/windows_mock/run_windows_mock_tests.sh")

add_setup_button=W.Button(description="+ Setup command",button_style="info")
add_test_button=W.Button(description="+ Test",button_style="info")
add_setup_button.on_click(lambda _: add_setup())
add_test_button.on_click(lambda _: add_test())

run_button=W.Button(description="▶ Run validation",button_style="success",layout=W.Layout(width="100%",height="44px"))
status=W.HTML("<b>Status:</b> Idle")
summary=W.VBox()
logs=W.Accordion(children=[])
report=W.Textarea(value="",placeholder="Complete report will appear here. Select/copy the text.",layout=W.Layout(width="100%",height="300px"))
download=W.Output(layout=W.Layout(width="100%"))
workspace=pathlib.Path("/content/tessera-validation-workspace")

def redact(text, token):
    text=str(text or "")
    if token: text=text.replace(token,"[REDACTED]")
    return text
def clip(text, limit=12000):
    text=str(text or "")
    if len(text)<=limit: return text
    return text[:limit//2]+"\n...[OUTPUT TRIMMED]...\n"+text[-limit//2:]
def run_shell(command,cwd,env,timeout):
    started=time.monotonic()
    try:
        p=subprocess.run(command,cwd=str(cwd),env=env,shell=True,executable="/bin/bash",
                         text=True,capture_output=True,timeout=timeout)
        output=(p.stdout or "")+("\n[STDERR]\n"+p.stderr if p.stderr else "")
        return p.returncode,round(time.monotonic()-started,2),output
    except subprocess.TimeoutExpired as exc:
        return 124,round(time.monotonic()-started,2),f"TIMEOUT ({timeout}s)\n{exc.stdout or ''}\n{exc.stderr or ''}"
    except Exception as exc:
        return -1,round(time.monotonic()-started,2),f"{type(exc).__name__}: {exc}"

def run_validation(_):
    run_button.disabled=True
    run_button.description="Running…"
    status.value="<b>Status:</b> Validating configuration…"
    summary.children=()
    logs.children=()
    report.value=""
    with download: clear_output()
    started=time.monotonic()
    issues=[]; setup_results=[]; test_results=[]
    token=None; cloned=False; commit="unknown"; setup_blocked=False
    try:
        url=repo.value.strip()
        branch=ref.value.strip()
        if not re.fullmatch(r"https://github.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?",url):
            raise ValueError("Only a GitHub HTTPS repository URL is accepted. Do not embed credentials.")
        if clone_timeout.value<1 or test_timeout.value<1 or depth.value<0:
            raise ValueError("Timeout values must be positive; depth cannot be negative.")
        specs=[]
        for idx,item in enumerate(test_rows,1):
            name=item["name"].value.strip(); command=item["command"].value.strip()
            needs=[n.strip() for n in item["needs"].value.split(",") if n.strip()]
            if name and command: specs.append({"name":name,"command":command,"needs":needs})
            elif name or command: raise ValueError(f"Test row {idx}: name and command are both required.")
        if not specs: raise ValueError("Add at least one complete test.")
        setup_commands=[item["field"].value.strip() for item in setup_rows if item["field"].value.strip()]
    except Exception as exc:
        issues.append(("Configuration",str(exc)))
        specs=[]; setup_commands=[]

    if not issues:
        shutil.rmtree(workspace,ignore_errors=True)
        env=os.environ.copy()
        env["GIT_TERMINAL_PROMPT"]="0"
        env["GCM_INTERACTIVE"]="Never"
        askdir=None
        try:
            if userdata is not None and secret_name.value.strip():
                try: token=userdata.get(secret_name.value.strip())
                except Exception as exc: issues.append(("Secret",f"Could not read Colab Secret: {type(exc).__name__}"))
            if token:
                askdir=pathlib.Path(tempfile.mkdtemp(prefix="tessera-askpass-"))
                ask=askdir/"askpass.sh"
                ask.write_text('#!/bin/sh\ncase "$1" in *Username*) echo "$TESSERA_GIT_USER";; *) echo "$TESSERA_GIT_TOKEN";; esac\n')
                ask.chmod(0o700)
                env.update({"GIT_ASKPASS":str(ask),"TESSERA_GIT_USER":git_user.value or "x-access-token","TESSERA_GIT_TOKEN":token})
            status.value="<b>Status:</b> Cloning repository…"
            clone_cmd=["git","clone"]
            if depth.value: clone_cmd += ["--depth",str(depth.value)]
            if branch: clone_cmd += ["--single-branch","--branch",branch]
            clone_cmd += [url,str(workspace)]
            cp=subprocess.run(clone_cmd,env=env,text=True,capture_output=True,timeout=clone_timeout.value)
            if cp.returncode:
                issues.append(("Clone",redact(clip(cp.stderr or cp.stdout),token)))
            else:
                cloned=True
                commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=workspace,text=True).strip()
        except subprocess.TimeoutExpired:
            issues.append(("Clone",f"Clone timed out after {clone_timeout.value} seconds."))
        except Exception as exc:
            issues.append(("Clone",f"{type(exc).__name__}: {redact(str(exc),token)}"))
        finally:
            if askdir: shutil.rmtree(askdir,ignore_errors=True)
            # Never forward credentials to setup/test subprocesses.
            for key in ("GIT_ASKPASS","TESSERA_GIT_USER","TESSERA_GIT_TOKEN"):
                env.pop(key,None)
            token_for_redaction=token
            token=None

        if cloned:
            for i,command in enumerate(setup_commands,1):
                status.value=f"<b>Status:</b> Setup {i}/{len(setup_commands)}…"
                code,duration,out=run_shell(command,workspace,env,test_timeout.value)
                record={"name":f"Setup {i}","command":command,"exit":code,"duration":duration,"output":clip(redact(out,token_for_redaction))}
                setup_results.append(record)
                if code:
                    issues.append((record["name"],record["output"] or f"exit={code}"))
                    setup_blocked=True
                    if not continue_on_error.value: break
            for spec in specs:
                if setup_blocked:
                    test_results.append({"name":spec["name"],"status":"SKIP","exit":None,"duration":0,"command":spec["command"],"output":"Skipped: setup command failed."})
                    continue
                missing=[tool for tool in spec["needs"] if shutil.which(tool) is None]
                if missing:
                    test_results.append({"name":spec["name"],"status":"SKIP","exit":None,"duration":0,"command":spec["command"],"output":"Missing executables: "+", ".join(missing)})
                    continue
                status.value=f"<b>Status:</b> Running {html.escape(spec['name'])}…"
                code,duration,out=run_shell(spec["command"],workspace,env,test_timeout.value)
                record={"name":spec["name"],"status":"PASS" if code==0 else "FAIL","exit":code,"duration":duration,"command":spec["command"],"output":clip(redact(out,token_for_redaction))}
                test_results.append(record)
                if code:
                    issues.append((spec["name"],record["output"] or f"exit={code}"))
                    if not continue_on_error.value: break
        else:
            test_results=[{"name":s["name"],"status":"SKIP","exit":None,"duration":0,"command":s["command"],"output":"Skipped: clone failed."} for s in specs]

    passed=sum(x["status"]=="PASS" for x in test_results)
    failed=sum(x["status"]=="FAIL" for x in test_results)
    skipped=sum(x["status"]=="SKIP" for x in test_results)
    overall="FAIL" if issues or failed or not cloned else ("PARTIAL" if skipped else ("PASS" if passed else "FAIL"))
    elapsed=round(time.monotonic()-started,2)
    report_lines=["TESSERA PORTABLE VALIDATION REPORT",f"Overall: {overall}",f"Repository: {repo.value}",f"Ref: {ref.value}",f"Commit: {commit}",f"Timestamp UTC: {datetime.datetime.now(datetime.timezone.utc).isoformat()}",f"Environment: {platform.platform()} | Python {platform.python_version()}",f"Duration: {elapsed}s",f"Results: {passed} PASS / {failed} FAIL / {skipped} SKIP",""]
    report_lines.append("SETUP:")
    if not setup_results: report_lines.append("(none)")
    for x in setup_results:
        report_lines.extend([f"[{'PASS' if x['exit']==0 else 'FAIL'}] {x['name']} | exit={x['exit']} | {x['duration']}s",f"COMMAND: {x['command']}",x['output'],""])
    report_lines.append("TESTS:")
    if not test_results: report_lines.append("(none)")
    for x in test_results:
        report_lines.extend([f"[{x['status']}] {x['name']} | exit={x['exit']} | {x['duration']}s",f"COMMAND: {x['command']}",x['output'],""])
    report_lines.append("ISSUES:")
    if not issues: report_lines.append("(none)")
    for stage,detail in issues: report_lines.extend([f"--- {stage} ---",detail,""])
    full_report="\n".join(report_lines)
    report.value=full_report
    report_path=pathlib.Path("/content/PROJECT_VALIDATION_REPORT.txt")
    report_path.write_text(full_report,encoding="utf-8")
    with download:
        clear_output()
        display(FileLink(str(report_path), result_html="<a href=\"PROJECT_VALIDATION_REPORT.txt\" download>⬇ Download report</a>"))
    if cloned and not keep_workspace.value: shutil.rmtree(workspace,ignore_errors=True)

    cards=[W.HTML(f"<b>{overall}</b> · {passed} PASS · {failed} FAIL · {skipped} SKIP · {elapsed}s")]
    cards.extend(W.HTML(f"<b>{html.escape(x['status'])}</b> — {html.escape(x['name'])} · {x['duration']}s · exit {x['exit']}") for x in test_results)
    summary.children=tuple(cards)
    log_items=[]
    for x in setup_results+test_results:
        body=W.Textarea(value=x["output"],layout=W.Layout(width="100%",height="180px"))
        title=(("PASS" if x.get("exit")==0 else "FAIL") if "exit" in x and x.get("exit") is not None else x.get("status","SKIP"))+" · "+x["name"]
        log_items.append(body)
    logs.children=tuple(log_items)
    for i,title in enumerate([(("PASS" if x.get("exit")==0 else "FAIL") if "exit" in x and x.get("exit") is not None else x.get("status","SKIP"))+" · "+x["name"] for x in setup_results+test_results]):
        logs.set_title(i,title)
    status.value=f"<b>Status:</b> Finished — {overall}"
    run_button.disabled=False
    run_button.description="▶ Run validation"

run_button.on_click(run_validation)

header=W.HTML("<h2>🧪 Tessera Test Console</h2><p>Add or remove setup steps and tests. Configure below and run. Only execute trusted code.</p>")
project=W.VBox([repo,ref,secret_name,git_user,W.HBox([depth,clone_timeout,test_timeout],layout=W.Layout(flex_flow="row wrap")),keep_workspace,continue_on_error])
setup_panel=W.VBox([W.HTML("<b>Setup commands</b>"),setup_box,add_setup_button])
tests_panel=W.VBox([W.HTML("<b>Tests</b>"),test_box,add_test_button])
for panel in (project,setup_panel,tests_panel):
    panel.layout.width="100%"
display(header,project,setup_panel,tests_panel,run_button,status,summary,W.HTML("<b>Logs</b>"),logs,W.HTML("<b>Full report (select text to copy)</b>"),report,download)
